In [ ]:
# Customer Purchase Behavior Analysis Using Machine Learning

**Assignment:** 2 – Machine Learning Model Development and Performance Evaluation

**Objective:** To develop, train, and evaluate machine learning models to predict invoice revenue using customer purchase transaction data.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")


Libraries imported successfully!


In [ ]:

from google.colab import files

uploaded = files.upload()


Saving Customer_Purchase_Cleaned.csv to Customer_Purchase_Cleaned.csv
Saving Online Retail.xlsx to Online Retail.xlsx


In [ ]:

# Load the uploaded dataset
filename = next(iter(uploaded))
df = pd.read_csv(filename)

# Display basic information
print("Dataset loaded successfully!")
print("Number of rows and columns:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

print("\nDataset information:")
df.info()


Dataset loaded successfully!
Number of rows and columns: (524878, 9)

Column names:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'Revenue']

First five rows:


/tmp/ipykernel_1207/2538753610.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 524878 entries, 0 to 524877
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    524878 non-null  object 
 1   StockCode    524878 non-null  object 
 2   Description  524878 non-null  object 
 3   Quantity     524878 non-null  int64  
 4   InvoiceDate  524878 non-null  object 
 5   UnitPrice    524878 non-null  float64
 6   CustomerID   392692 non-null  float64
 7   Country      524878 non-null  object 
 8   Revenue      524878 non-null  float64
dtypes: float64(3), int64(1), object(5)
memory usage: 36.0+ MB


In [ ]:

# Step 6: Check data quality

print("Missing values in each column:")
print(df.isnull().sum())

print("\nNumber of duplicate rows:")
print(df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

print("\nSummary statistics:")
display(df.describe(include="all").T)


Missing values in each column:
InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     132186
Country             0
Revenue             0
dtype: int64

Number of duplicate rows:
0

Data types:
InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID     float64
Country         object
Revenue        float64
dtype: object

Summary statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
InvoiceNo,524878.0,19962.0,573585.0,1114.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,524878,3922,85123A,2253,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,524878,4026,WHITE HANGING HEART T-LIGHT HOLDER,2311,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,524878.0,NaN,NaN,NaN,10.6166,156.280031,1.0,1.0,4.0,11.0,80995.0
InvoiceDate,524878,18499,2011-10-31 14:41:00,1114,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UnitPrice,524878.0,NaN,NaN,NaN,3.922573,36.093028,0.001,1.25,2.08,4.13,13541.33
CustomerID,392692.0,NaN,NaN,NaN,15287.843865,1713.539549,12346.0,13955.0,15150.0,16791.0,18287.0
Country,524878,38,United Kingdom,479985,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Revenue,524878.0,NaN,NaN,NaN,20.275399,271.693566,0.001,3.9,9.92,17.7,168469.6


In [ ]:

# Step 7: Prepare the dataset for Machine Learning

# Convert InvoiceDate into datetime format
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"], errors="coerce"
)

# Remove rows with missing essential values
df = df.dropna(
    subset=[
        "InvoiceNo", "InvoiceDate",
        "Quantity", "UnitPrice", "Country"
    ]
)

# Keep valid positive sales transactions
df = df[
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0)
].copy()

# Create Revenue if it is missing
if "Revenue" not in df.columns:
    df["Revenue"] = df["Quantity"] * df["UnitPrice"]

print("Data preprocessing completed!")
print("Rows and columns:", df.shape)
print("Missing values in essential columns:")
print(df[[
    "InvoiceNo", "InvoiceDate",
    "Quantity", "UnitPrice", "Country"
]].isnull().sum())


Data preprocessing completed!
Rows and columns: (524878, 9)
Missing values in essential columns:
InvoiceNo      0
InvoiceDate    0
Quantity       0
UnitPrice      0
Country        0
dtype: int64


In [ ]:

# Step 8: Feature engineering

# Extract purchase time features
df["Hour"] = df["InvoiceDate"].dt.hour
df["DayOfWeek"] = df["InvoiceDate"].dt.dayofweek
df["Month"] = df["InvoiceDate"].dt.month

# Group transaction lines by invoice
invoice_df = df.groupby("InvoiceNo").agg(
    InvoiceRevenue=("Revenue", "sum"),
    TotalQuantity=("Quantity", "sum"),
    NumberOfLines=("StockCode", "count"),
    UniqueProducts=("StockCode", "nunique"),
    Country=("Country", "first"),
    Hour=("Hour", "first"),
    DayOfWeek=("DayOfWeek", "first"),
    Month=("Month", "first")
).reset_index()

print("Invoice-level dataset created successfully!")
print("Dataset shape:", invoice_df.shape)
display(invoice_df.head())


Invoice-level dataset created successfully!
Dataset shape: (19962, 9)


,InvoiceNo,InvoiceRevenue,TotalQuantity,NumberOfLines,UniqueProducts,Country,Hour,DayOfWeek,Month
0,536365,139.12,40,7,7,United Kingdom,8,2,12
1,536366,22.20,12,2,2,United Kingdom,8,2,12
2,536367,278.73,83,12,12,United Kingdom,8,2,12
3,536368,70.05,15,4,4,United Kingdom,8,2,12
4,536369,17.85,3,1,1,United Kingdom,8,2,12


In [ ]:

# Step 8: Feature Engineering

# Extract time-related features
df["Hour"] = df["InvoiceDate"].dt.hour
df["DayOfWeek"] = df["InvoiceDate"].dt.dayofweek
df["Month"] = df["InvoiceDate"].dt.month

# Combine product rows into invoice-level records
invoice_df = df.groupby("InvoiceNo").agg(
    InvoiceRevenue=("Revenue", "sum"),
    TotalQuantity=("Quantity", "sum"),
    NumberOfLines=("StockCode", "count"),
    UniqueProducts=("StockCode", "nunique"),
    Country=("Country", "first"),
    Hour=("Hour", "first"),
    DayOfWeek=("DayOfWeek", "first"),
    Month=("Month", "first")
).reset_index()

print("Invoice-level dataset created successfully!")
print("Dataset shape:", invoice_df.shape)

display(invoice_df.head())


NameError: name 'df' is not defined

In [ ]:

# Step 9: Define features and target

# Target: the value we want to predict
X = invoice_df[
    [
        "TotalQuantity",
        "NumberOfLines",
        "UniqueProducts",
        "Country",
        "Hour",
        "DayOfWeek",
        "Month"
    ]
]

y = invoice_df["InvoiceRevenue"]

print("Features prepared successfully!")
print("Number of input features:", X.shape[1])
print("Target variable:", y.name)

# Split data into training and testing sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("\nTraining records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])


NameError: name 'invoice_df' is not defined

In [ ]:

# Re-create invoice-level dataset

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"], errors="coerce"
)

df = df.dropna(
    subset=[
        "InvoiceNo", "InvoiceDate",
        "Quantity", "UnitPrice", "Country"
    ]
)

df = df[
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0)
].copy()

if "Revenue" not in df.columns:
    df["Revenue"] = df["Quantity"] * df["UnitPrice"]

df["Hour"] = df["InvoiceDate"].dt.hour
df["DayOfWeek"] = df["InvoiceDate"].dt.dayofweek
df["Month"] = df["InvoiceDate"].dt.month

invoice_df = df.groupby("InvoiceNo").agg(
    InvoiceRevenue=("Revenue", "sum"),
    TotalQuantity=("Quantity", "sum"),
    NumberOfLines=("StockCode", "count"),
    UniqueProducts=("StockCode", "nunique"),
    Country=("Country", "first"),
    Hour=("Hour", "first"),
    DayOfWeek=("DayOfWeek", "first"),
    Month=("Month", "first")
).reset_index()

print("Invoice dataset created successfully!")
print("Shape:", invoice_df.shape)

display(invoice_df.head())


NameError: name 'pd' is not defined

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")


Libraries imported successfully!


In [ ]:

print("Dataset loaded:", "df" in globals())

if "df" in globals():
    print("Dataset shape:", df.shape)
    display(df.head())


Dataset loaded: False


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

uploaded = files.upload()


Saving Customer_Purchase_Cleaned.csv to Customer_Purchase_Cleaned.csv
Saving Online Retail.xlsx to Online Retail.xlsx


In [ ]:

# Step 10: Prepare data for Machine Learning

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Separate numerical and categorical columns
numeric_features = [
    "TotalQuantity",
    "NumberOfLines",
    "UniqueProducts",
    "Hour",
    "DayOfWeek",
    "Month"
]

categorical_features = ["Country"]

# Preprocessing for numerical data
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Preprocessing for categorical data
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both preprocessing methods
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocessing pipeline created successfully!")
print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)


Preprocessing pipeline created successfully!
Numerical features: ['TotalQuantity', 'NumberOfLines', 'UniqueProducts', 'Hour', 'DayOfWeek', 'Month']
Categorical features: ['Country']


In [ ]:

# Assignment 2: Feature Engineering

# Convert InvoiceDate into datetime format
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"], errors="coerce"
)

# Remove rows missing essential values
df = df.dropna(
    subset=[
        "InvoiceNo", "InvoiceDate",
        "Quantity", "UnitPrice", "Country"
    ]
)

# Keep positive sales transactions
df = df[
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0)
].copy()

# Create Revenue if it is missing
if "Revenue" not in df.columns:
    df["Revenue"] = df["Quantity"] * df["UnitPrice"]

# Extract time features
df["Hour"] = df["InvoiceDate"].dt.hour
df["DayOfWeek"] = df["InvoiceDate"].dt.dayofweek
df["Month"] = df["InvoiceDate"].dt.month

# Aggregate transactions by invoice
invoice_df = df.groupby("InvoiceNo").agg(
    InvoiceRevenue=("Revenue", "sum"),
    TotalQuantity=("Quantity", "sum"),
    NumberOfLines=("StockCode", "count"),
    UniqueProducts=("StockCode", "nunique"),
    Country=("Country", "first"),
    Hour=("Hour", "first"),
    DayOfWeek=("DayOfWeek", "first"),
    Month=("Month", "first")
).reset_index()

print("Invoice dataset created successfully!")
print("Dataset shape:", invoice_df.shape)
display(invoice_df.head())


NameError: name 'df' is not defined

In [ ]:

filename = next(iter(uploaded))
df = pd.read_csv(filename)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())


Dataset loaded successfully!
Shape: (524878, 9)
Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'Revenue']


/tmp/ipykernel_2780/1729365990.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [ ]:

# Assignment 2 - Feature Engineering

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"], errors="coerce"
)

df = df.dropna(
    subset=[
        "InvoiceNo", "InvoiceDate",
        "Quantity", "UnitPrice", "Country"
    ]
)

df = df[
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0)
].copy()

if "Revenue" not in df.columns:
    df["Revenue"] = df["Quantity"] * df["UnitPrice"]

df["Hour"] = df["InvoiceDate"].dt.hour
df["DayOfWeek"] = df["InvoiceDate"].dt.dayofweek
df["Month"] = df["InvoiceDate"].dt.month

invoice_df = df.groupby("InvoiceNo").agg(
    InvoiceRevenue=("Revenue", "sum"),
    TotalQuantity=("Quantity", "sum"),
    NumberOfLines=("StockCode", "count"),
    UniqueProducts=("StockCode", "nunique"),
    Country=("Country", "first"),
    Hour=("Hour", "first"),
    DayOfWeek=("DayOfWeek", "first"),
    Month=("Month", "first")
).reset_index()

print("Invoice dataset created successfully!")
print("Dataset shape:", invoice_df.shape)
display(invoice_df.head())


Invoice dataset created successfully!
Dataset shape: (19962, 9)


,InvoiceNo,InvoiceRevenue,TotalQuantity,NumberOfLines,UniqueProducts,Country,Hour,DayOfWeek,Month
0,536365,139.12,40,7,7,United Kingdom,8,2,12
1,536366,22.20,12,2,2,United Kingdom,8,2,12
2,536367,278.73,83,12,12,United Kingdom,8,2,12
3,536368,70.05,15,4,4,United Kingdom,8,2,12
4,536369,17.85,3,1,1,United Kingdom,8,2,12


In [ ]:

# Step 12: Prepare features and target

from sklearn.model_selection import train_test_split

X = invoice_df[
    [
        "TotalQuantity",
        "NumberOfLines",
        "UniqueProducts",
        "Country",
        "Hour",
        "DayOfWeek",
        "Month"
    ]
]

y = invoice_df["InvoiceRevenue"]

# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Data split completed successfully!")
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Number of features:", X_train.shape[1])


Data split completed successfully!
Training records: 15969
Testing records: 3993
Number of features: 7


In [ ]:

# Step 13: Train Linear Regression Model

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Define feature columns
numeric_features = [
    "TotalQuantity",
    "NumberOfLines",
    "UniqueProducts",
    "Hour",
    "DayOfWeek",
    "Month"
]

categorical_features = ["Country"]

# Preprocess numerical features
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Preprocess categorical features
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Create and train the model
linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)

# Predict invoice revenue
linear_predictions = linear_model.predict(X_test)

# Evaluate the model
linear_mae = mean_absolute_error(y_test, linear_predictions)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_predictions))
linear_r2 = r2_score(y_test, linear_predictions)

print("LINEAR REGRESSION RESULTS")
print("-------------------------")
print("MAE:", round(linear_mae, 2))
print("RMSE:", round(linear_rmse, 2))
print("R-squared (R2):", round(linear_r2, 4))


LINEAR REGRESSION RESULTS
-------------------------
MAE: 216.94
RMSE: 1102.16
R-squared (R2): 0.8483


In [ ]:

# Step 14: Train Decision Tree Regressor

from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Create the Decision Tree model
tree_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(
        max_depth=10,
        random_state=42
    ))
])

# Train the model
tree_model.fit(X_train, y_train)

# Predict invoice revenue
tree_predictions = tree_model.predict(X_test)

# Evaluate performance
tree_mae = mean_absolute_error(y_test, tree_predictions)
tree_rmse = np.sqrt(mean_squared_error(y_test, tree_predictions))
tree_r2 = r2_score(y_test, tree_predictions)

print("DECISION TREE REGRESSION RESULTS")
print("--------------------------------")
print("MAE:", round(tree_mae, 2))
print("RMSE:", round(tree_rmse, 2))
print("R-squared (R2):", round(tree_r2, 4))


DECISION TREE REGRESSION RESULTS
--------------------------------
MAE: 226.46
RMSE: 1619.72
R-squared (R2): 0.6724


In [ ]:

# Step 15: Train Random Forest Regressor

from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Create the model
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    ))
])

# Train the model
rf_model.fit(X_train, y_train)

# Predict invoice revenue
rf_predictions = rf_model.predict(X_test)

# Evaluate performance
rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))
rf_r2 = r2_score(y_test, rf_predictions)

print("RANDOM FOREST REGRESSION RESULTS")
print("--------------------------------")
print("MAE:", round(rf_mae, 2))
print("RMSE:", round(rf_rmse, 2))
print("R-squared (R2):", round(rf_r2, 4))


RANDOM FOREST REGRESSION RESULTS
--------------------------------
MAE: 216.6
RMSE: 1854.97
R-squared (R2): 0.5703


In [ ]:
comparison_df = pd.DataFrame({
    "Model": ["Linear Regression", "Decision Tree", "Random Forest"],
    "MAE": [linear_mae, tree_mae, rf_mae],
    "RMSE": [linear_rmse, tree_rmse, rf_rmse],
    "R2 Score": [linear_r2, tree_r2, rf_r2]
})

display(comparison_df.round(4))

,Model,MAE,RMSE,R2 Score
0,Linear Regression,216.9368,1102.1587,0.8483
1,Decision Tree,226.4650,1619.7232,0.6724
2,Random Forest,216.6005,1854.9709,0.5703


In [ ]:
feature_names = rf_model.named_steps["preprocessor"].get_feature_names_out()

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_model.named_steps["model"].feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance", ascending=False
)

display(importance_df.head(10).round(4))

,Feature,Importance
0,num__TotalQuantity,0.7743
2,num__UniqueProducts,0.0546
3,num__Hour,0.0487
1,num__NumberOfLines,0.0399
5,num__Month,0.0349
4,num__DayOfWeek,0.0325
41,cat__Country_United Kingdom,0.0056
6,cat__Country_Australia,0.0021
35,cat__Country_Singapore,0.0017
30,cat__Country_Netherlands,0.0015


In [ ]:
import joblib

joblib.dump(linear_model, "linear_regression_model.joblib")
joblib.dump(tree_model, "decision_tree_model.joblib")
joblib.dump(rf_model, "random_forest_model.joblib")

print("All 3 trained models saved successfully!")


All 3 trained models saved successfully!


In [ ]:
from google.colab import files

files.download("linear_regression_model.joblib")
files.download("decision_tree_model.joblib")
files.download("random_forest_model.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Assignment 2: Machine Learning Model Development and Performance Evaluation

**Project Title:** Customer Purchase Behavior Analysis

**Course:** AD25C01 – Exploratory Data Analysis

**Department:** Artificial Intelligence and Data Science

**Institution:** SRM TRP Engineering College

## 1. Introduction
This project analyzes customer purchase transaction data and develops machine learning models to estimate invoice revenue.

## 2. Problem Statement
The objective is to estimate the total revenue of an invoice using transaction characteristics such as quantity, number of product lines, unique products, country, and purchase date and time.

## 3. Dataset Description
The dataset contains customer purchase transaction records. The data was cleaned before model development. Individual transaction records were aggregated by invoice number to create invoice-level data.

## 4. Data Preprocessing
The preprocessing steps include handling missing values, removing invalid or non-positive quantities and prices, extracting date and time features, and encoding categorical data.

## 5. Machine Learning Models
Three regression models were developed:
- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor

## 6. Model Evaluation
The models were evaluated using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and the R-squared (R²) score.

## 7. Model Comparison
The performance of the three models was compared to identify the most suitable model for this dataset.

## 8. Feature Importance
Random Forest feature importance was used to identify the features that contributed most to the model's predictions.

## 9. Conclusion
The project demonstrates the application of regression algorithms to customer purchase transaction data. The final conclusion will be based on the measured model performance.

## 10. Limitations and Future Improvements
The model estimates revenue using transaction characteristics that may only be known after purchase. Future work could explore features available before a purchase and test additional models.

# Customer Purchase Behavior Analysis
## Assignment 2: Machine Learning Model Development and Performance Evaluation

### Slide 1: Title
- Project title
- Student name and roll number
- Department of Artificial Intelligence and Data Science
- SRM TRP Engineering College

### Slide 2: Introduction
- Overview of customer purchase transaction data
- Importance of analyzing invoice revenue
- Application of machine learning

### Slide 3: Problem Statement and Objective
- Estimate invoice revenue using transaction characteristics
- Develop and compare three regression models

### Slide 4: Dataset Description
- Transaction records and dataset features
- Invoice number, quantity, unit price, country, and revenue
- Invoice-level aggregation

### Slide 5: Data Preprocessing
- Handle missing values
- Remove invalid quantities and prices
- Extract hour, day of week, and month
- Encode categorical features

### Slide 6: Machine Learning Workflow
- Feature selection
- 80% training data and 20% testing data
- Model training and evaluation

### Slide 7: Model 1 — Linear Regression
- Model overview
- Training and prediction
- Evaluation using MAE, RMSE, and R²

### Slide 8: Model 2 — Decision Tree Regressor
- Model overview
- Training and prediction
- Evaluation results

### Slide 9: Model 3 — Random Forest Regressor
- Model overview
- Training and prediction
- Evaluation results

### Slide 10: Model Comparison
- Compare MAE, RMSE, and R²
- Identify the best-performing model using the actual results

### Slide 11: Feature Importance and Discussion
- Top features identified by Random Forest
- Interpretation of the results
- Limitations and future improvements

### Slide 12: Conclusion
- Summary of the experiments
- Best model and justification
- Future scope
- Thank you



## Model Evaluation and Comparison

Three regression models were evaluated using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and the R-squared (R²) score.

### Results and Discussion

**Linear Regression:** This model achieved an MAE of 216.9368, RMSE of 1102.1587, and R² score of 0.8483. It obtained the highest R² score and lowest RMSE among the three models, indicating the strongest overall performance on the test data.

**Decision Tree Regressor:** This model achieved an MAE of 226.4650, RMSE of 1619.7232, and R² score of 0.6724. Its performance was lower than Linear Regression on all three evaluation measures.

**Random Forest Regressor:** This model achieved an MAE of 216.6005, RMSE of 1854.9709, and R² score of 0.5703. It recorded the lowest MAE by a small margin, but its higher RMSE and lower R² score indicate weaker overall performance in this experiment.

### Best Model Selection

Linear Regression is selected as the best overall model because it achieved the highest R² score and the lowest RMSE on the test set. Random Forest had a marginally lower MAE, so the final choice depends on the evaluation priorities. Based on the overall results, Linear Regression is preferred.

### Conclusion

The experiment demonstrates that Linear Regression performed better overall than the Decision Tree and Random Forest models for estimating invoice revenue on this test set. Further evaluation using cross-validation and additional features could help confirm these findings.

## Feature Importance Analysis

Random Forest feature importance was used to identify the features that contributed most to predicting invoice revenue.

### Key Findings

- **TotalQuantity (0.7743):** This is the most important feature, accounting for approximately 77.43% of the model's total feature importance.
- **UniqueProducts (0.0546):** The number of distinct products in an invoice is the second most important feature.
- **Hour (0.0487):** The hour of the transaction contributes to the model's predictions.
- **NumberOfLines (0.0399):** The number of product lines in an invoice also contributes to revenue estimation.
- **Month (0.0349) and DayOfWeek (0.0325):** These time-related features have smaller contributions.
- **Country features:** The United Kingdom has an importance score of 0.0056, while Australia, Singapore, and the Netherlands have relatively small importance scores.

### Interpretation

TotalQuantity is substantially more important than the other features in this Random Forest model. This suggests that the total quantity of products purchased is strongly associated with the model's revenue predictions. The other features contribute smaller amounts of predictive information.

Feature importance indicates how the model uses features; it does not prove that a feature causes higher revenue. These results apply to the Random Forest model and do not automatically explain the other regression models.

## Conclusion

This project developed and evaluated three machine learning regression models to estimate invoice revenue from customer purchase transaction data. Linear Regression, Decision Tree Regressor, and Random Forest Regressor were evaluated using MAE, RMSE, and R².

Based on the experimental results, Linear Regression performed best overall, achieving an R² score of 0.8483 and an RMSE of 1102.1587. Random Forest achieved a slightly lower MAE, while Decision Tree showed weaker overall performance. The Random Forest feature-importance analysis identified TotalQuantity as the most influential feature.

The project demonstrates how machine learning can support the analysis of customer purchase transactions and invoice revenue.

## Limitations

1. The model was evaluated using a single train-test split.
2. The dataset may contain unusual transactions that affect model performance.
3. TotalQuantity, NumberOfLines, and UniqueProducts are known from the completed invoice, so the model estimates revenue using purchase characteristics rather than predicting customer spending before a purchase.
4. Random Forest feature importance describes the model's reliance on features and does not establish causation.
5. Model performance may differ on new data.

## Future Improvements

1. Apply cross-validation to evaluate model stability.
2. Tune model hyperparameters to improve performance.
3. Investigate outliers and unusual invoice values.
4. Test additional regression algorithms.
5. Explore features available before a purchase for genuine advance revenue forecasting.

In [27]:
from sklearn.metrics import mean_squared_error

linear_mse = mean_squared_error(y_test, linear_predictions)
tree_mse = mean_squared_error(y_test, tree_predictions)
rf_mse = mean_squared_error(y_test, rf_predictions)

print("Linear Regression MSE:", round(linear_mse, 4))
print("Decision Tree MSE:", round(tree_mse, 4))
print("Random Forest MSE:", round(rf_mse, 4))

Linear Regression MSE: 1214753.9056
Decision Tree MSE: 2623503.2723
Random Forest MSE: 3440917.0507
